In [2]:
import pandas as pd
import re
from ollama import chat, ChatResponse
from tqdm import tqdm  # 用於顯示進度條

In [3]:
# --- 設定 ---
FILE_PATH = '../data/all.xlsx'     # 資料集路徑
MODEL_NAME = 'gemma3:12b'  # 模型名稱

In [4]:
# --- 函數：呼叫 Ollama (使用官方庫) ---
def query_ollama(prompt, model=MODEL_NAME):
    try:
        # 使用 ollama.chat 直接呼叫
        response: ChatResponse = chat(
            model=model,
            messages=[
                {'role': 'user', 'content': prompt}
            ],
            # 您可以在此加入 options，例如 temperature
            options={
                'temperature': 0.0  # 讓輸出結果穩定
            }
        )
        # 回傳模型的內容
        return response.message.content
    except Exception as e:
        print(f"Error calling Ollama: {e}")
        return ""

In [5]:
# --- 函數：解析模型輸出 ---
def extract_answer(text):
    """
    從模型的回答中提取選項數字 (1, 2, 3, 4)。
    DeepSeek R1 會輸出 <think>...</think> 思考過程，需要過濾。
    """
    if not text:
        return None

    # 1. 移除 <think> 標籤內的內容 (DeepSeek R1 特有)
    # flags=re.DOTALL 確保 . 可以匹配換行符號
    text_cleaned = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()
    
    # 2. 策略 A: 尋找明確的 "Answer: X" 格式 (Prompt 中有要求)
    match = re.search(r'Answer:\s*([1-4])', text_cleaned, re.IGNORECASE)
    if match:
        return int(match.group(1))
    
    # 3. 策略 B: 如果沒有標準格式，尋找最後出現的數字 1-4
    # 這是一個備用方案
    numbers = re.findall(r'\b([1-4])\b', text_cleaned)
    if numbers:
        return int(numbers[-1])
        
    return None  # 解析失敗

In [6]:
# --- 主程式 ---
# 1. 讀取資料
print(f"正在讀取 {FILE_PATH}...")
try:
    df = pd.read_excel(FILE_PATH)
except Exception as e:
    print(f"讀取檔案失敗: {e}")

# 檢查必要的欄位
required_columns = ['ID', 'Article', 'Question', 'Option1', 'Option2', 'Option3', 'Option4', 'Answer']
if not all(col in df.columns for col in required_columns):
    print(f"錯誤: Excel 檔案缺少必要的欄位。需要: {required_columns}")

正在讀取 ../data/all.xlsx...


In [7]:
df

,ID,Article,Question,Option1,Option2,Option3,Option4,Answer
0,1,明年元旦起，交通部新規定，汽車輪胎胎紋深度將納入定期檢驗項目之一，一旦深度未達1.6公厘，及...,汽車胎紋未到達多少公厘將會被吊銷牌照?,1.5公厘,1.4公厘,1.3公厘,1.6公厘,4
1,2,塑化劑是一種「環境荷爾蒙」，又稱內分泌干擾物，指會干擾生物體內分泌之外因性化學物質。若長時間...,若長時間使用哪一種塑化劑會使男嬰生殖器發育不良、女童性早熟，以及成年男性精蟲減少等問題。,DEHP,DINP,DNOP,DIDP,1
2,3,紐西蘭的海岸邊近日出現數百具「小藍企鵝」的屍體，專家指出，這些企鵝都是被活活餓死的，會有企鵝...,使此處海水變暖的原因為何?,洋流,全球暖化,反聖嬰現象,核能電廠排放廢水,3
3,4,綜合報導，科學期刊《當代生物學》昨天刊登一份美澳聯合研究，研究指出，近年澳洲大堡礁北部的新出...,為甚麼雄性海龜出生率降低，雌性海龜出生率則升高?,機率問題,全球暖化,環境賀爾蒙影響,當地環境影響,2
4,5,花蓮強震發生後，是否還可能發生更大的地震？中央大學地科系教授馬國鳳警告，昨晚花蓮強震應該是主...,這次花蓮大地震主要是哪一地震斷層所影響?,奇美斷層,米崙斷層,玉里斷層,嶺頂斷層,2
...,...,...,...,...,...,...,...,...
8545,8546,臺北市位於木柵地區的福德坑垃圾掩埋場變身「台北能源之丘」，成為全台首座由垃圾場轉型的太陽能發...,下列何者為台北能源之丘計畫?,全國第一個完工發電的垃圾掩埋場轉型風力發電廠,全國第一個完工發電的垃圾掩埋場轉型太陽能電廠,全國第一個完工發電的垃圾掩埋場轉型水力發電廠,全國第一個完工發電的垃圾掩埋場轉型核能電廠,2
8546,8547,臺北市位於木柵地區的福德坑垃圾掩埋場變身「台北能源之丘」，成為全台首座由垃圾場轉型的太陽能發...,垃圾掩埋場轉型的太陽能電廠，預估ㄧ年發電量最高可達幾萬度?,200萬,300萬,2000萬,20000萬,1
8547,8548,春雨不足，加上核一廠機二號機六月初停擺，今天夏天供電情況相當嚴峻，尤其水力供電要看天吃飯，如...,什麼力供電要看天氣?,火力,電力,水力,風力,3
8548,8549,春雨不足，加上核一廠機二號機六月初停擺，今天夏天供電情況相當嚴峻，尤其水力供電要看天吃飯，如...,台電董事長是誰?,朱文章,朱文,朱成,朱文成,4


In [8]:
# --- 主程式 ---
def main():
    correct_count = 0
    total_questions = len(df)
    results = []

    print(f"開始使用 {MODEL_NAME} 處理 {total_questions} 個問題...")

    # 2. 遍歷每一列進行預測
    for index, row in tqdm(df.iterrows(), total=total_questions):
        article = row['Article']
        question = row['Question']
        # 建立選項字典方便取用
        options = {
            1: row['Option1'],
            2: row['Option2'],
            3: row['Option3'],
            4: row['Option4']
        }
        actual_answer = row['Answer']
        #print(actual_answer)

        # 建構 Prompt
        # 這裡的設計是為了讓 DeepSeek R1 能夠先思考再回答，並最後輸出固定格式
        prompt = f"""
Read the following article and answer the multiple-choice question.

Article:
{article}

Question:
{question}

Options:
1. {options[1]}
2. {options[2]}
3. {options[3]}
4. {options[4]}

Instruction:
Analyze the article and select the best option. 
Finally, output the answer in the exact format: "Answer: X", where X is the option number (1, 2, 3, or 4).
Do not output anything else after the answer.
"""
        # 呼叫模型
        response_text = query_ollama(prompt)
        
        # 解析答案
        predicted_answer = extract_answer(response_text)
        
        # 比對結果
        is_correct = False
        try:
            # 處理 Excel 讀入的答案可能是浮點數或字串的情況
            actual_answer_int = int(actual_answer)
            if predicted_answer is not None and predicted_answer == actual_answer_int:
                correct_count += 1
                is_correct = True
            else:
                print("Article:", article)
                print("Question:", question)
                print("Option:", "1.", row['Option1'], "2.", row['Option2'], "3.", row['Option3'], "4.", row['Option4'])
                print("Answer:", actual_answer)
                print("Predicted Answer:", predicted_answer)
                print('-'*30)
        except ValueError:
            pass # 忽略無效的標準答案格式

        # 儲存詳細結果
        results.append({
            "ID": row['ID'],
            "Actual": actual_answer,
            "Predicted": predicted_answer,
            "Correct": is_correct,
            "Model_Response": response_text
        })

        #print(predicted_answer)
        if (index+1) % 500 == 0:
            print(correct_count, " / ", (index+1), " = ", 100.0*correct_count/(index+1), "%")
            break

    # 3. 計算並輸出正確率
    accuracy = (correct_count / total_questions) * 100 if total_questions > 0 else 0
    print("\n" + "="*30)
    print(f"處理完成！")
    print(f"總題數: {total_questions}")
    print(f"答對題數: {correct_count}")
    print(f"正確率 (Accuracy): {accuracy:.2f}%")
    print("="*30)

    # 4. 將結果存檔
    output_file = "result_analysis.xlsx"
    pd.DataFrame(results).to_excel(output_file, index=False)
    print(f"詳細結果已儲存至 {output_file}")

if __name__ == "__main__":
    main()

開始使用 gemma3:12b 處理 8550 個問題...


  1%|▍                                                                             | 52/8550 [01:31<3:52:49,  1.64s/it]

Article: 在這種極端低溫下，粒子物質波的波長很長、粒子與粒子間的物質波有很大部分重疊，形成一種稱為玻色─愛因斯坦凝聚的物態，也就是玻色子粒子冷卻到接近絕對零度時，呈現出一種氣態、零黏度的超流體狀態，此時我們熟悉的物理規則退到幕後，量子力學開始接管，我們會看到原子行為不再像粒子，反而更像波浪。
Question: 何種情況下會形成玻色─愛因斯坦凝聚的物態?
Option: 1. 宇宙背景輻射溫度 3K 2. 氣體原子雲溫度冷卻至0.0001K 3. 極端低溫下 4. 極端高溫下
Answer: 3
Predicted Answer: 2
------------------------------


  1%|▉                                                                            | 107/8550 [02:58<3:45:54,  1.61s/it]

Article: 該網友表示，自己4月去日本遊玩，原本沒打算替人代買商品，不料卻因打卡洩漏了行蹤，而一位友人主動私訊請原PO幫忙代購「送爸媽」的物品，原PO勉為其難的答應。沒想到，原PO將價值共2萬多元的商品帶回台灣後，對方竟因「價格太貴」將所有東西取消，讓他氣得大罵「他拜託我買了...白七星3條、黑七星2條、櫻花裝2條、Wakamoto3罐、痠痛貼2盒、眼藥水3盒⋯等，所以每天你要我跟你回報價錢是回報好玩的嗎？」
Question: 請問文中黑七星買了幾條?
Option: 1. 2條 2. 3條 3. 4條 4. 5條
Answer: 2
Predicted Answer: 1
------------------------------


...

Article: 美杜莎是戈爾貢三姐妹中唯一的血肉之軀，所以珀耳修斯受到雅典娜和赫耳墨斯的指點，才成功砍下她的頭顱并殺死她，珀耳修斯将美杜莎的頭顱裝在特製的皮囊中，隨後飛上天空逃走了。在天空中還遭遇狂風，在經過利比亞沙漠時，皮囊不小心傾斜一下，從美杜莎頭顱上滴下的點點鮮血落到地上，變成各型各色的毒蛇。珀耳修斯借助美杜莎頭顱的力量殺死海怪，解救安德洛墨達，還將菲纽斯和他的士兵們變成石像
Question: 美杜莎的頭是誰砍下的?
Option: 1. 雅典娜 2. 珀耳修斯 3. 赫耳墨斯 4. 宙斯
Answer: 1
Predicted Answer: 2
------------------------------


  5%|████▏                                                                        | 460/8550 [12:53<3:29:54,  1.56s/it]

Article: 大老二手牌大小比較
先比點數：2>A>K>Q>J>10>9>8>7>6>5>4>3
後比花色：
台灣大老二玩法：♠ > ♥ > ♦ > ♣。
Question: 下列何者最小?
Option: 1. ♠2 2. ♠3 3. ♦2 4. ♣K
Answer: 2
Predicted Answer: 4
------------------------------


  6%|████▍                                                                        | 499/8550 [13:55<3:44:42,  1.67s/it]

477  /  500  =  95.4 %

處理完成！
總題數: 8550
答對題數: 477
正確率 (Accuracy): 5.58%
詳細結果已儲存至 result_analysis.xlsx


In [ ]:
# --- 主程式：只測一筆 ---
def main():
    # 1. 讀取資料
    try:
        df = pd.read_excel(FILE_PATH)
        if len(df) == 0:
            print("錯誤：Excel 檔案是空的。")
            return
    except Exception as e:
        print(f"讀取檔案失敗: {e}")
        return

    # 2. 取出第一筆資料 (Index 0)
    print(f"--- 正在測試第一筆資料 (ID: {df.iloc[0]['ID']}) ---")
    row = df.iloc[0]
    
    article = row['Article']
    question = row['Question']
    options = {
        1: row['Option1'],
        2: row['Option2'],
        3: row['Option3'],
        4: row['Option4']
    }
    actual_answer = row['Answer']

    # 3. 建構 Prompt
    prompt = f"""
Read the following article and answer the multiple-choice question.

Article:
{article}

Question:
{question}

Options:
1. {options[1]}
2. {options[2]}
3. {options[3]}
4. {options[4]}

Instruction:
Analyze the article and select the best option. 
Finally, output the answer in the exact format: "Answer: X", where X is the option number (1, 2, 3, or 4).
Do not output anything else after the answer.
"""
    
    print("\n[生成的 Prompt 預覽 (前 200 字)]:")
    print(prompt[:200] + "...\n")

    # 4. 執行預測
    response_text = query_ollama(prompt)
    predicted_answer = extract_answer(response_text)

    # 5. 顯示詳細結果
    print("="*40)
    print("【模型完整回應 (含思考過程)】:")
    print(response_text)
    print("="*40)
    
    print(f"【解析後的預測答案】: {predicted_answer}")
    print(f"【Excel 中的正確答案】: {actual_answer}")

    try:
        if int(predicted_answer) == int(actual_answer):
            print("✅ 結果判定: 答對了！")
        else:
            print("❌ 結果判定: 答錯了。")
    except:
        print("⚠️ 無法判定 (答案格式可能不是數字)")

if __name__ == "__main__":
    main()